# Phase 5 · Submission S5 — Ensemble of the Phase 5 models (probability averaging)

**Task:** Legal Statute Identification (100 statute IDs, multi-label). **Primary metric:** Macro-F1.

**Idea:** S1 (A + LORMA), S2 (C + layer-wise FT) and S3 (D + adapters) are different Phase-1 architectures that make partly different mistakes; averaging their per-statute probabilities is a simple, well-known way to reduce variance (it was already S4's teacher and beat every single model on dev: 0.3119 vs 0.2981). No training happens here — the notebook reads the dev and test logits each run saved (`outputs/teacher_logits.pt`, S4: `outputs/student_logits.pt`).

**Method**
1. Load the dev/test logits of every available run and check they belong to exactly the dev and test cases of `dev.json` / `test.json` (same order) and the same 100 statute IDs.
2. Score a few fixed combinations on dev — each single model, S1+S2+S3, S1+S2+S3+S4, and S1 counted twice with S2+S3 — with the same per-statute thresholds and cross-fitted dev estimate as S1–S4 (thresholds tuned on one half of dev, scored on the other).
3. Pick the combination with the best cross-fitted dev Macro-F1; tune its per-statute thresholds on the whole dev set; predict test (at least one statute per case) → `submission.zip`.

Only Phase-1-based models and the competition data are used; dev is used for selection and thresholds only, test labels are never read.

**Before running (Kaggle)**
- **CPU is enough** (no GPU) · Add-ons → Secrets: **`HF`** · Internet on.
- Add Input → datasets: **`Phase5_S1_Dataset`**, **`Phase5_S2_Dataset`**, **`Phase5_S3_Dataset`** and (optional) **`Phase5_S4_Dataset`**.
- Expected time ≈ 3–5 minutes.

Outputs: `/kaggle/working/P5_S5_Ensemble/` — upload `submission.zip`.

In [ ]:
# Cell 1 — Imports and random seed (no GPU needed).
import json, glob, zipfile
from pathlib import Path
from collections import Counter
import numpy as np
import torch
from huggingface_hub import snapshot_download
from kaggle_secrets import UserSecretsClient

SEED = 42
np.random.seed(SEED)
print("PyTorch", torch.__version__)

In [ ]:
# Cell 2 — Paths, the runs to combine, the combinations to try and the threshold grid.
RUN_NAME = "P5_S5_Ensemble"
WORK_DIR = Path("/kaggle/working") / RUN_NAME
DATA_DIR = Path("/kaggle/working/data")
WORK_DIR.mkdir(parents=True, exist_ok=True); DATA_DIR.mkdir(parents=True, exist_ok=True)
HF_REPO_ID = "Exploration-Lab/CS-781-Capstone"
RUNS = {"S1": ("P5_S1_A_LORMA", "teacher_logits.pt"), "S2": ("P5_S2_C_LSTM_LayerwiseFT", "teacher_logits.pt"),
        "S3": ("P5_S3_D_TCN_Adapters", "teacher_logits.pt"), "S4": ("P5_S4_KD_Ensemble_to_BEncoder_LoRA", "student_logits.pt")}
REQUIRED = ("S1", "S2", "S3")                     # S4 is optional
COMBINATIONS = {"S1": {"S1": 1}, "S2": {"S2": 1}, "S3": {"S3": 1}, "S4": {"S4": 1},
                "S1+S2+S3": {"S1": 1, "S2": 1, "S3": 1}, "S1+S2+S3+S4": {"S1": 1, "S2": 1, "S3": 1, "S4": 1},
                "2·S1+S2+S3": {"S1": 2, "S2": 1, "S3": 1}}
THRESHOLD_GRID = np.round(np.arange(0.05, 0.951, 0.025), 3)
print("Outputs:", WORK_DIR)

In [ ]:
# Cell 3 — Download only dev.json (labels for selection / thresholds) and test.json (ids) of Phase_5_LSI.
snapshot_download(repo_id=HF_REPO_ID, repo_type="dataset", token=UserSecretsClient().get_secret("HF"), local_dir=str(DATA_DIR),
                  allow_patterns=["Phase_5_LSI/Train/dev.json", "Phase_5_LSI/Test/test.json"])
P5_DIR = DATA_DIR / "Phase_5_LSI"
raw_dev = json.load(open(P5_DIR / "Train/dev.json"))
test_ids = [str(d["id"]) for d in json.load(open(P5_DIR / "Test/test.json"))]
dev_ids = [str(d["id"]) for d in raw_dev]
print(f"dev {len(dev_ids):,} cases | test {len(test_ids):,} cases")

In [ ]:
# Cell 4 — Load each run's saved logits and check they match these dev / test cases and the same statute IDs.
runs = {}
for short, (run, filename) in RUNS.items():
    paths = sorted(glob.glob(f"/kaggle/input/**/{run}/outputs/{filename}", recursive=True))
    if not paths:
        assert short not in REQUIRED, f"Add the output dataset of {run} ({short}) as an input."
        print(f"{short}: not attached — skipped")
        continue
    t = torch.load(paths[0], map_location="cpu", weights_only=False)
    assert t["ids"]["dev"] == dev_ids and t["ids"]["test"] == test_ids, f"{short}: dev/test cases differ"
    runs[short] = t
    print(f"{short}: {run} | test setting {t['setting']} | best epoch {t['best_epoch']}")
LABEL_IDS = runs["S1"]["label_ids"]
assert all(t["label_ids"] == LABEL_IDS for t in runs.values()), "Statute IDs differ between runs"
LABEL_INDEX = {l: j for j, l in enumerate(LABEL_IDS)}
Y_dev = np.zeros((len(raw_dev), len(LABEL_IDS)), dtype=np.float32)
for i, d in enumerate(raw_dev):
    Y_dev[i, [LABEL_INDEX[int(l)] for l in d["labels"]]] = 1.0
print(f"{len(LABEL_IDS)} statute IDs | dev labels per case {Y_dev.sum(1).mean():.2f}")

In [ ]:
# Cell 5 — The same multi-label metrics and threshold rules as S1–S4 (per-statute thresholds, cross-fitted dev estimate).
def per_label_f1(Y, pred):
    tp = (Y * pred).sum(0); fp = ((1 - Y) * pred).sum(0); fn = (Y * (1 - pred)).sum(0)
    return 2 * tp / np.maximum(2 * tp + fp + fn, 1)

def predict_sets(P, thresholds):
    pred = (P >= thresholds).astype(np.float32)
    empty = pred.sum(1) == 0
    pred[np.where(empty)[0], np.argmax(P[empty] / thresholds, axis=1)] = 1.0
    return pred

def all_scores(Y, pred):
    tp, fp, fn = (Y * pred).sum(), ((1 - Y) * pred).sum(), (Y * (1 - pred)).sum()
    tp_l = (Y * pred).sum(0)
    return {"macro_f1": float(per_label_f1(Y, pred).mean()), "micro_f1": float(2 * tp / max(2 * tp + fp + fn, 1)),
            "subset_accuracy": float((pred == Y).all(1).mean()),
            "macro_precision": float((tp_l / np.maximum(pred.sum(0), 1)).mean()),
            "macro_recall": float((tp_l / np.maximum(Y.sum(0), 1)).mean()), "labels_per_case": float(pred.sum(1).mean())}

def tune_thresholds(P, Y):
    best_t, best_f = np.full(P.shape[1], 0.5), np.full(P.shape[1], -1.0)
    for t in THRESHOLD_GRID:
        f = per_label_f1(Y, (P >= t).astype(np.float32))
        better = f > best_f
        best_t[better], best_f[better] = t, f[better]
    return best_t

def crossfit_predictions(P, Y):
    halves = np.array_split(np.random.RandomState(SEED).permutation(len(P)), 2)
    pred = np.zeros_like(P)
    for a, b in ((0, 1), (1, 0)):
        pred[halves[b]] = predict_sets(P[halves[b]], tune_thresholds(P[halves[a]], Y[halves[a]]))
    return pred

def combined_probs(weights, split):
    total = sum(weights.values())
    return sum(w * torch.sigmoid(runs[r][split].float()).numpy() for r, w in weights.items()) / total
print("Metrics ready.")

In [ ]:
# Cell 6 — Score every available combination on dev (cross-fitted per-statute thresholds) and pick the best.
results = {}
for name, weights in COMBINATIONS.items():
    if not all(r in runs for r in weights):
        continue
    P = combined_probs(weights, "dev")
    results[name] = {"crossfit": all_scores(Y_dev, crossfit_predictions(P, Y_dev)),
                     "full_dev": all_scores(Y_dev, predict_sets(P, tune_thresholds(P, Y_dev)))}
    r = results[name]["crossfit"]
    print(f"  {name:12s}: Macro-F1 {r['macro_f1']:.4f} | Macro-P {r['macro_precision']:.4f} | Macro-R {r['macro_recall']:.4f} | "
          f"Micro-F1 {r['micro_f1']:.4f} | subset acc {r['subset_accuracy']:.4f} | labels/case {r['labels_per_case']:.2f}")
BEST = max(results, key=lambda k: results[k]["crossfit"]["macro_f1"])
print(f"Best on dev: {BEST} ({results[BEST]['crossfit']['macro_f1']:.4f}; S1 alone {results['S1']['crossfit']['macro_f1']:.4f})")

In [ ]:
# Cell 7 — Final thresholds on the whole dev set, test predictions, format checks, submission.zip and a run summary.
P_dev = combined_probs(COMBINATIONS[BEST], "dev")
THRESHOLDS = tune_thresholds(P_dev, Y_dev)
pred_test = predict_sets(combined_probs(COMBINATIONS[BEST], "test"), THRESHOLDS)
prediction = [{"id": i, "pred": [int(LABEL_IDS[j]) for j in np.where(p == 1)[0]]} for i, p in zip(test_ids, pred_test)]
valid = set(LABEL_IDS)
assert len(prediction) == len(test_ids) and len({p["id"] for p in prediction}) == len(prediction)
assert all(p["pred"] and len(set(p["pred"])) == len(p["pred"]) and set(p["pred"]) <= valid and
           all(isinstance(x, int) for x in p["pred"]) for p in prediction)
json.dump(prediction, open(WORK_DIR / "prediction.json", "w"))
with zipfile.ZipFile(WORK_DIR / "submission.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(WORK_DIR / "prediction.json", arcname="prediction.json")
with zipfile.ZipFile(WORK_DIR / "submission.zip") as zf:
    assert zf.namelist() == ["prediction.json"]
sizes = np.array([len(p["pred"]) for p in prediction])
json.dump({"run": RUN_NAME, "chosen": BEST, "weights": COMBINATIONS[BEST], "dev": results,
           "thresholds": {str(LABEL_IDS[j]): float(THRESHOLDS[j]) for j in range(len(LABEL_IDS))},
           "test_statutes_per_case": float(sizes.mean())}, open(WORK_DIR / "run_summary.json", "w"), indent=2)
print(f"submission.zip ({BEST}): {len(prediction):,} cases | statutes per case mean {sizes.mean():.2f} "
      f"(dev gold {Y_dev.sum(1).mean():.2f}), max {sizes.max()} | distinct statutes {len({x for p in prediction for x in p['pred']})}")